## Curricular Unit: Information Integration and Analytic Data Processing
## Professora: Márcia Barros

### Group 5

### Nuno Lages 66091
### Nuno Rosado 66104
### Diogo Carvalho 66123

---

### Imports

In [1]:
import json
import os
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import ast
import unicodedata
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Extraction

In [2]:
# ===============================
# CONFIGURATION
# ===============================

RUN_ENV = "local"     # options: "colab" or "local"

# Colab configuration
COLAB_BASE_DIR = "/content/drive/My Drive/FCUL-MEI/Semestre_2/IPAI/Projeto"

# Local configuration
LOCAL_BASE_DIR =   "./"  # change to your local folder if needed

if RUN_ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')

    base_path = Path(COLAB_BASE_DIR)

elif RUN_ENV == "local":
    base_path = Path(LOCAL_BASE_DIR)

else:
    raise ValueError("RUN_ENV must be 'colab' or 'local'")

# Create base directory if needed
base_path.mkdir(parents=True, exist_ok=True)
# Create data directory if needed
(base_path / "data").mkdir(parents=True, exist_ok=True)

# File paths
SONGS = base_path / "data" / "merged_songs.csv"
ARTISTS = base_path / "data" / "artists.csv"
REVIEWS = base_path / "data" / "pitchfork_reviews.csv"

# Dimension paths
dim_music_file = base_path / "data" / "dim_music.tsv"
dim_date_file = base_path / "data" / "dim_date.tsv"
dim_genre_file = base_path / "data" /"dim_genre.tsv"
dim_album_file = base_path / "data" /"dim_album.tsv"
dim_review_file = base_path / "data" / "dim_review.tsv"
dim_artist_file = base_path / "data" / "dim_artist.tsv"

# Fact table path
fact_music_file = base_path / "data" / "fact_music.tsv"

# Lookup files
music_lookup_file = base_path / "data" / "lk_music.json"
date_lookup_file = base_path / "data" / "lk_date.json"
genre_lookup_file = base_path / "data" / "lk_genre.json"
album_lookup_file = base_path / "data" / "lk_album.json"
review_lookup_file = base_path / "data" / "lk_review.json"
artist_lookup_file = base_path / "data" / "lk_artist.json"

# Transformation

In [3]:
songs_df = pd.read_csv(SONGS, sep=',')
artists_df = pd.read_csv(ARTISTS, sep=',')
reviews_df = pd.read_csv(REVIEWS, sep=';')

# Songs dataset cleaning
songs_df.info()
songs_df = songs_df.dropna(subset=['album_name'])
songs_df = songs_df.dropna(subset=['release_date'])
songs_df = songs_df.drop_duplicates(subset=['id'])
# Maintain only the entries that have a complete date
songs_df = songs_df[songs_df['release_date'].str.len() == 10]
songs_df = songs_df[~(songs_df['lyrics'].str.contains(r'\[Instrumental\]', na=False))]

# Artists dataset cleaning
artists_df.info()
artists_df = artists_df.dropna()
artists_df = artists_df.drop_duplicates(subset=['name'])

# Reviews dataset cleaning
reviews_df.info()
reviews_df = reviews_df.dropna()
reviews_df = reviews_df.drop_duplicates()

# Convert the list of genres into a real list
artists_df['genres'] = artists_df['genres'].apply(ast.literal_eval)

# Text normalization
songs_df['album_name'] = songs_df['album_name'].str.lower().str.strip()
reviews_df['album'] = reviews_df['album'].str.lower().str.strip()

songs_df['artists'] = songs_df['artists'].str.lower().str.strip()
reviews_df['artist'] = reviews_df['artist'].str.lower().str.strip()
artists_df['name'] = artists_df['name'].str.lower().str.strip()

songs_df['genre'] = songs_df['genre'].str.lower().str.strip()
reviews_df['genre'] = reviews_df['genre'].str.lower().str.strip()
artists_df['main_genre'] = artists_df['main_genre'].str.lower().str.strip()
artists_df['genres'] = artists_df['genres'].apply(
    lambda x: [g.lower().strip() for g in x]
)

# Convert Review date to datetime (DD/MM/AAAA)
reviews_df['review_date'] = pd.to_datetime(
    reviews_df['review_date'],
    errors='coerce'
)
reviews_df['review_date'] = reviews_df['review_date'].dt.strftime('%d/%m/%Y')


# Kepp only reviews of the existing albums (9012 Reviews)
albums = songs_df['album_name'].dropna().unique()
reviews_df = reviews_df[reviews_df['album'].isin(albums)]


# ===============================
# EXPORT CLEANED FILES
# ===============================

songs_df.to_csv(base_path / "data" / "clean_songs.csv", index=False)
artists_df.to_csv(base_path / "data" / "clean_artists.csv", index=False)
reviews_df.to_csv(base_path / "data" / "clean_reviews.csv", index=False)

print("\nCleaned files saved successfully.")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47472 entries, 0 to 47471
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   name                    47472 non-null  object 
 1   release_date            41217 non-null  object 
 2   id                      47472 non-null  object 
 3   album_name              47472 non-null  object 
 4   artists                 47472 non-null  object 
 5   danceability            47472 non-null  float64
 6   energy                  47472 non-null  float64
 7   key                     47472 non-null  int64  
 8   loudness                47472 non-null  float64
 9   mode                    47472 non-null  int64  
 10  speechiness             47472 non-null  float64
 11  acousticness            47472 non-null  float64
 12  instrumentalness        47472 non-null  float64
 13  liveness                47472 non-null  float64
 14  valence                 47472 non-null

# Loading

In [4]:
# ---------- Helpers ----------

def load_lookup_table(dimension_name: str):
    """Load the natural key -> surrogate key mapping for a dimension."""
    file_name = f"lut_{dimension_name}.json"
    if os.path.isfile(file_name):
        with open(file_name, "r", encoding="utf-8") as f:
            lookup = json.load(f)
        # JSON values may already be integers, but enforce it just in case.
        lookup = {k: int(v) for k, v in lookup.items()}
    else:
        lookup = {}

    next_surrogate_key = (max(lookup.values()) + 1) if lookup else 1
    return lookup, next_surrogate_key


def save_lookup_table(dimension_name: str, lookup_table: dict):
    """Persist the lookup table to disk."""
    file_name = f"lut_{dimension_name}.json"
    with open(file_name, "w", encoding="utf-8") as f:
        json.dump(lookup_table, f, ensure_ascii=False, indent=2)


def save_dimension(df: pd.DataFrame, file_name: str):
    """Append new rows to a dimension TSV file, creating the header if needed."""
    write_header = not os.path.isfile(file_name)
    mode = "w" if write_header else "a"

    if not df.empty:
        df.to_csv(file_name, sep="\t", index=False, mode=mode, header=write_header)


def save_fact(df: pd.DataFrame, file_name: str):
    """Append fact rows to the fact TSV file, creating the header if needed."""
    write_header = not os.path.isfile(file_name)
    mode = "w" if write_header else "a"
    df.to_csv(file_name, sep="\t", index=False, mode=mode, header=write_header)


def assign_surrogate_keys(natural_keys: pd.Series, lookup_table: dict, next_surrogate_key: int):
    """Assign surrogate keys to a series of natural keys.

    Returns:
        key_series: surrogate key for each row in natural_keys
        new_rows_df: dataframe with only the newly discovered natural keys
        updated_lookup_table
        updated_next_surrogate_key
    """
    new_keys = [key for key in pd.unique(natural_keys) if key not in lookup_table]

    if new_keys:
        new_mapping = {key: sk for sk, key in enumerate(new_keys, start=next_surrogate_key)}
        lookup_table.update(new_mapping)
        next_surrogate_key += len(new_keys)
        new_rows_df = pd.DataFrame({
            "natural_key": new_keys,
            "surrogate_key": [lookup_table[key] for key in new_keys]
        })
    else:
        new_rows_df = pd.DataFrame(columns=["natural_key", "surrogate_key"])

    key_series = natural_keys.map(lookup_table).astype(int)
    return key_series, new_rows_df, lookup_table, next_surrogate_key

### Music Dimension

In [5]:
def build_music_dimension(df_songs: pd.DataFrame):
    lookup, next_sk = load_lookup_table("music")

    music_keys, new_songs, lookup, next_sk = assign_surrogate_keys(
        df_songs["id"], lookup, next_sk
    )

    dim_music = new_songs.rename(columns={
        "surrogate_key": "MUSIC_KEY",
        "natural_key": "ID"
    })

    if not dim_music.empty:
        music_attributes = (
            df_songs[["id", "name", "lyrics"]]
            .drop_duplicates(subset=["id"])
        )

        dim_music = (
            dim_music.merge(music_attributes, left_on="ID", right_on="id", how="left")
            [["MUSIC_KEY", "ID", "name", "lyrics"]]
        )

    save_dimension(dim_music, dim_music_file)
    save_lookup_table("music", lookup)

    return music_keys, dim_music, lookup

In [6]:
music_keys, dim_music, lookup_song = build_music_dimension(songs_df)
print("Dimensão de Música construída e guardada!")

Dimensão de Música construída e guardada!


### Date Dimension

In [7]:
def get_season(month):
    if month in [12, 1, 2]: return "Winter"
    if month in [3, 4, 5]: return "Spring"
    if month in [6, 7, 8]: return "Summer"
    return "Autumn"

def build_date_dimension(df_songs: pd.DataFrame, df_reviews: pd.DataFrame):
    lookup, next_sk = load_lookup_table("date")

    # 1. Extrair datas das reviews
    rev_dates = pd.to_datetime(df_reviews["review_date"], errors='coerce').dropna()

    # 2. Extrair datas das músicas
    song_dates = pd.to_datetime(df_songs["release_date"], errors='coerce').dropna()

    # 3. Concatenar ambas e obter apenas a parte da data (YYYY-MM-DD) única
    all_dates = pd.concat([rev_dates, song_dates]).dt.date.unique()

    unique_dates_str = [str(d) for d in all_dates]

    df_dates = pd.DataFrame({"date_raw": unique_dates_str})

    # 3. Atribuir Surrogate Keys
    date_keys, new_dates, lookup, next_sk = assign_surrogate_keys(
        df_dates["date_raw"], lookup, next_sk
    )

    # 4. Renomear colunas base
    dim_date = new_dates.rename(columns={
        "surrogate_key": "Date_ID",
        "natural_key": "Complete Date"
    })

    if not dim_date.empty:
        dt = pd.to_datetime(dim_date["Complete Date"])

        dim_date["Full Date (Extended Format)"] = dt.dt.strftime('%B %d, %Y')
        dim_date["Standard American Date"] = dt.dt.strftime('%m/%d/%Y')
        dim_date["Day of Month"] = dt.dt.day
        dim_date["Day of Year"] = dt.dt.dayofyear
        dim_date["Day of Week"] = dt.dt.day_name()
        dim_date["Week of Year"] = dt.dt.isocalendar().week.astype(int)
        dim_date["Month Description"] = dt.dt.month_name()
        dim_date["Month Abbreviation"] = dt.dt.strftime('%b')
        dim_date["Month Number"] = dt.dt.month
        dim_date["Quarter"] = dt.dt.quarter
        dim_date["Semester"] = dt.dt.month.apply(lambda x: 1 if x <= 6 else 2)
        dim_date["Year"] = dt.dt.year
        dim_date["Decade"] = (dt.dt.year // 10) * 10
        dim_date["Season"] = dt.dt.month.apply(get_season)

        ordered_cols = [
            "Date_ID", "Complete Date", "Full Date (Extended Format)",
            "Standard American Date", "Day of Month", "Day of Year",
            "Day of Week", "Week of Year", "Month Description",
            "Month Abbreviation", "Month Number", "Quarter",
            "Semester", "Year", "Decade", "Season"
        ]

        dim_date = dim_date[ordered_cols]

    save_dimension(dim_date, dim_date_file)
    save_lookup_table("date", lookup)

    return date_keys, dim_date, lookup

date_keys, dim_date, lookup_date = build_date_dimension(songs_df, reviews_df)

### Artist Dimension

In [8]:
def build_artist_dimension(df_artists: pd.DataFrame):
    lookup, next_sk = load_lookup_table("artist")

    artist_keys, new_artists, lookup, next_sk = assign_surrogate_keys(
        df_artists["name"], lookup, next_sk
    )

    dim_artist = new_artists.rename(columns={
        "surrogate_key": "Artist_ID",
        "natural_key": "Name"
    })

    if not dim_artist.empty:
        dim_artist = dim_artist.merge(
            df_artists[["name", "followers", "popularity"]],
            left_on="Name", right_on="name", how="left"
        ).drop(columns=["name"]).rename(columns={
            "followers": "Followers",
            "popularity": "Artist Popularity"
        })

        cols = ["Artist_ID", "Name", "Followers", "Artist Popularity"]
        dim_artist = dim_artist[cols]

    save_dimension(dim_artist, dim_artist_file)
    save_lookup_table("artist", lookup)

    return artist_keys, dim_artist, lookup

In [9]:
artist_keys, dim_artist, lookup_artist = build_artist_dimension(artists_df)

In [26]:
dim_artist = dim_artist.drop_duplicates(subset=['Artist_ID'])

### Genre Dimension

In [11]:
def build_genre_dimension(df_songs: pd.DataFrame):
    lookup, next_sk = load_lookup_table("genre")

    unique_genres = df_songs["genre"].dropna().unique()

    genre_keys, new_genres, lookup, next_sk = assign_surrogate_keys(
        pd.Series(unique_genres), lookup, next_sk
    )

    dim_genre = new_genres.rename(columns={
        "surrogate_key": "Genre_ID",
        "natural_key": "Genre name"
    })

    if not dim_genre.empty:
        dim_genre = dim_genre[["Genre_ID", "Genre name"]]

    save_dimension(dim_genre, dim_genre_file)
    save_lookup_table("genre", lookup)

    return genre_keys, dim_genre, lookup

In [12]:
genre_keys, dim_genre, lookup_genre = build_genre_dimension(songs_df)

### Album e Review Dimensions

In [13]:
def build_album_review_dimensions(df_songs: pd.DataFrame, df_reviews: pd.DataFrame):
    lk_alb, next_sk_alb = load_lookup_table("album")

    df_songs['alb_nat_key'] = df_songs['album_name'].astype(str) + "||" + df_songs['artists'].astype(str)

    album_keys, new_albs, lk_alb, next_sk_alb = assign_surrogate_keys(
        df_songs["alb_nat_key"], lk_alb, next_sk_alb
    )

    dim_album = pd.DataFrame({
        "ALB_NAT_KEY": list(lk_alb.keys()),
        "Album_ID": list(lk_alb.values())
    })

    album_names = df_songs[["alb_nat_key", "album_name"]].drop_duplicates("alb_nat_key")
    dim_album = dim_album.merge(album_names, left_on="ALB_NAT_KEY", right_on="alb_nat_key", how="left")

    dim_album = dim_album.rename(columns={"album_name": "Name"})[["Album_ID", "Name"]]

    dim_album.to_csv(dim_album_file, sep="\t", index=False)
    save_lookup_table("album", lk_alb)


    lk_rev, next_sk_rev = load_lookup_table("review")
    df_reviews['rev_nat_key'] = df_reviews['album'].astype(str) + "||" + df_reviews['artist'].astype(str)

    review_keys, new_revs, lk_rev, next_sk_rev = assign_surrogate_keys(
        df_reviews['rev_nat_key'], lk_rev, next_sk_rev
    )

    dim_review = new_revs.rename(columns={"surrogate_key": "Review_ID", "natural_key": "REV_NAT_KEY"})

    if not dim_review.empty:
        df_reviews['Album_ID'] = df_reviews['rev_nat_key'].map(lk_alb)

        dim_review = dim_review.merge(
            df_reviews[["rev_nat_key", "Album_ID", "reviewer", "label", "summary", "review", "score"]],
            left_on="REV_NAT_KEY", right_on="rev_nat_key", how="left"
        ).rename(columns={
            "score": "Classification",
            "reviewer": "Reviewer",
            "label": "Label",
            "summary": "Summary",
            "review": "Review"
        })

    cols_to_drop = ["REV_NAT_KEY", "rev_nat_key"]
    dim_review = dim_review.drop(columns=[c for c in cols_to_drop if c in dim_review.columns], errors='ignore')
    
    # Garante que não há colunas duplicadas por erro de merge (acontece se houver nomes iguais)
    dim_review = dim_review.loc[:, ~dim_review.columns.duplicated()]

    save_dimension(dim_review, dim_review_file)
    save_lookup_table("review", lk_rev)

    return lk_alb, lk_rev, dim_album, dim_review

In [14]:
lk_album, lk_review, dim_album, dim_review = build_album_review_dimensions(songs_df, reviews_df)
#dim_review = dim_review.drop(columns=['rev_nat_key'])

In [15]:
dim_review = dim_review.drop_duplicates(subset=['Review_ID'])

### Fact Table

In [16]:
def build_music_fact(df_songs, df_reviews, lks):
    # 1. Criar uma cópia para não alterar o DataFrame original
    fact = df_songs.copy()

    # 2. Mapeamento das Chaves Estrangeiras (FKs)
    # Music_ID e Artist_ID
    fact["Music_ID"] = fact["id"].map(lks['music'])
    fact["Artist_ID"] = fact["artists"].map(lks['artist'])

    # Genre_ID
    fact["Genre_ID"] = fact["genre"].map(lks['genre'])

    # Album_ID (usando a mesma lógica de chave composta da dimensão album)
    fact["album_nat_key"] = fact["album_name"].astype(str) + "||" + fact["artists"].astype(str)
    fact["Album_ID"] = fact["album_nat_key"].map(lks['album'])

    # 3. Mapeamento da Date_ID (Data da Música)
    # Garantir que a data está em formato string 'YYYY-MM-DD' para o lookup
    fact["release_date_str"] = pd.to_datetime(fact["release_date"], errors='coerce').dt.date.astype(str)
    fact["Date_ID"] = fact["release_date_str"].map(lks['date'])

    # 4. Tratamento de Métricas e Renomeação
    cols_mapping = {
        "duration_ms": "Duration",
        "danceability": "Danceability",
        "energy": "Energy",
        "loudness": "Loudness",
        "speechiness": "Speechiness",
        "acousticness": "Acousticness",
        "instrumentalness": "Instrumentalness",
        "liveness": "Liveness",
        "popularity": "Music popularity"
    }
    fact = fact.rename(columns=cols_mapping)

    # 5. Cálculo de Word Count (Baseado na coluna lyrics)
    fact["Word count"] = fact["lyrics"].fillna("").apply(lambda x: len(str(x).split()))

    # 6. Seleção final de colunas e limpeza
    final_cols = ["Music_ID", "Artist_ID", "Album_ID", "Date_ID", "Genre_ID"] + \
                 list(cols_mapping.values()) + ["Word count"]

    # Remover linhas onde não foi possível mapear a Música (integridade referencial)
    fact_music = fact[final_cols].dropna(subset=["Music_ID"])

    # Converter IDs para inteiros para evitar o formato float (ex: 1.0)
    id_cols = ["Music_ID", "Artist_ID", "Album_ID", "Date_ID", "Genre_ID"]
    fact_music[id_cols] = fact_music[id_cols].fillna(0).astype(int)

    # 7. Gravar o ficheiro
    save_fact(fact_music, fact_music_file)
    return fact_music

# --- EXECUÇÃO ---

# Certifique-se de que os lookups estão atualizados
lks = {
    'music': lookup_song,
    'artist': lookup_artist,
    'genre': lookup_genre,
    'album': lk_album,
    'date': lookup_date
}

fact_music = build_music_fact(songs_df, reviews_df, lks)
print("Tabela de Factos construída com sucesso!")

Tabela de Factos construída com sucesso!


In [22]:
import sqlalchemy
from sqlalchemy import create_engine

# Configurações de acesso
USER = 'postgres'
PASSWORD = 'mestrado'
HOST = '127.0.0.1'
PORT = '5432'
DB_NAME = 'postgres'

# Criar o motor de conexão
engine = create_engine(f'postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}')

In [23]:
def upload_to_postgress():
  try:
    with engine.connect() as conn:
        print("Ligação OK")
  except Exception as e:
    print(e)

upload_to_postgress()

Ligação OK


In [24]:
def sanitize_columns(df):
    df_clean = df.copy()
    df_clean.columns = df_clean.columns.str.lower().str.replace(' ', '_')
    return df_clean

def upload_to_postgres_robust():
    tables = {
        'dim_music': dim_music,
        'dim_artist': dim_artist,
        'dim_date': dim_date,
        'dim_genre': dim_genre,
        'dim_album': dim_album,
        'dim_review': dim_review,
        'fact_music': fact_music
    }
    
    print("A iniciar upload para o PostgreSQL")
    for name, df in tables.items():
        try:
            df_sql = sanitize_columns(df)
            df_sql.to_sql(name, engine, if_exists='replace', index=False)
            print(f"{name}: {len(df_sql)} linhas carregadas.")
        except Exception as e:
            print(f"Erro em {name}: {e}")

upload_to_postgres_robust()

A iniciar upload para o PostgreSQL
dim_music: 40593 linhas carregadas.
dim_artist: 69643 linhas carregadas.
dim_date: 6046 linhas carregadas.
dim_genre: 10 linhas carregadas.
dim_album: 6381 linhas carregadas.
dim_review: 6465 linhas carregadas.
fact_music: 81186 linhas carregadas.


## Indexs

In [27]:
def create_indexes():
    try:
        with engine.connect() as conn:
            print("A criar índices para otimização...")
            
            # 1. Índices nas Foreign Keys da Tabela de Factos
            conn.execute(sqlalchemy.text("CREATE INDEX idx_fact_music_id ON fact_music(music_id);"))
            conn.execute(sqlalchemy.text("CREATE INDEX idx_fact_artist_id ON fact_music(artist_id);"))
            conn.execute(sqlalchemy.text("CREATE INDEX idx_fact_album_id ON fact_music(album_id);"))
            conn.execute(sqlalchemy.text("CREATE INDEX idx_fact_date_id ON fact_music(date_id);"))
            conn.execute(sqlalchemy.text("CREATE INDEX idx_fact_genre_id ON fact_music(genre_id);"))
            
            # 2. Índices de Performance (Para buscas frequentes)
            conn.execute(sqlalchemy.text("CREATE INDEX idx_music_popularity ON fact_music(music_popularity);"))
            conn.execute(sqlalchemy.text("CREATE INDEX idx_review_score ON dim_review(classification);"))
            
            # 3. Se as dimensões foram criadas com to_sql(replace), elas não têm PKs definidas no SQL.
            conn.execute(sqlalchemy.text("ALTER TABLE dim_music ADD PRIMARY KEY (music_key);"))
            conn.execute(sqlalchemy.text("ALTER TABLE dim_artist ADD PRIMARY KEY (artist_id);"))
            conn.execute(sqlalchemy.text("ALTER TABLE dim_album ADD PRIMARY KEY (album_id);"))
            conn.execute(sqlalchemy.text("ALTER TABLE dim_date ADD PRIMARY KEY (date_id);"))
            conn.execute(sqlalchemy.text("ALTER TABLE dim_genre ADD PRIMARY KEY (genre_id);"))
            conn.execute(sqlalchemy.text("ALTER TABLE dim_review ADD PRIMARY KEY (review_id);"))

            conn.commit() 
            print("Todos os índices e PKs foram criados com sucesso")

    except Exception as e:
        print(f"Erro ao criar índices: {e}")

create_indexes()

A criar índices para otimização...
Erro ao criar índices: (psycopg2.errors.UniqueViolation) could not create unique index "dim_artist_pkey"
DETAIL:  Key (artist_id)=(85) is duplicated.

[SQL: ALTER TABLE dim_artist ADD PRIMARY KEY (artist_id);]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


## Extract Searchable Documents

In [21]:
fact_music = pd.read_csv('data/fact_music.tsv', sep='\t')
dim_music = pd.read_csv('data/dim_music.tsv', sep='\t')
dim_artist = pd.read_csv('data/dim_artist.tsv', sep='\t')
dim_album = pd.read_csv('data/dim_album.tsv', sep='\t')
dim_genre = pd.read_csv('data/dim_genre.tsv', sep='\t')
dim_date = pd.read_csv('data/dim_date.tsv', sep='\t')
dim_review = pd.read_csv('data/dim_review.tsv', sep='\t')

# Group reviews of the same album
reviews_grouped = dim_review.groupby('Album_ID').apply(
    lambda x: x[['Summary', 'Review']].rename(
        columns={'Summary': 'summary', 'Review': 'review'}
    ).to_dict('records')
).reset_index(name='reviews')

df = fact_music[['Music_ID', 'Artist_ID', 'Album_ID', 'Date_ID', 'Genre_ID', 'Word count']].copy()

# Add the dimensions to the df
df = df.merge(dim_music[['MUSIC_KEY', 'name', 'lyrics']], left_on='Music_ID', right_on='MUSIC_KEY', how='left')
df = df.merge(dim_artist[['Artist_ID', 'Name']], on='Artist_ID', how='left')
df = df.merge(dim_album[['Album_ID', 'Name']], on='Album_ID', how='left', suffixes=('', '_album'))
df = df.merge(dim_genre[['Genre_ID', 'Genre name']], on='Genre_ID', how='left')
df = df.merge(dim_date[['Date_ID', 'Complete Date']], on='Date_ID', how='left')
df = df.merge(reviews_grouped, on='Album_ID', how='left')

documents = []
for _, row in df.iterrows():
    doc = {
        "title": str(row['name']),
        "release_date": str(row['Complete Date']),
        "artist": {
            "name": str(row['Name'])
        },
        "album": {
            "name": str(row['Name_album'])
        },
        "genre": str(row['Genre name']),
        "content": {
            "lyrics": str(row['lyrics']),
            "word_count": int(row['Word count']) if pd.notnull(row['Word count']) else 0
        },
        "reviews": row['reviews']
    }
    documents.append(doc)

with open('jsons/searchable_documents.json', 'w', encoding='utf-8') as f:
    json.dump(documents, f, indent=2, ensure_ascii=False)

print(f"Sucesso! Foram gerados {len(documents)} documentos pesquisáveis.")

C:\Users\nunor\AppData\Local\Temp\ipykernel_33784\670339077.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  reviews_grouped = dim_review.groupby('Album_ID').apply(


Sucesso! Foram gerados 81544 documentos pesquisáveis.


### Measure Document

In [ ]:
fact_music = pd.read_csv('data/fact_music.tsv', sep='\t')
dim_review = pd.read_csv('data/dim_review.tsv', sep='\t')

# Agrupamento das reviews por álbum (classificação)
review_scores = dim_review.groupby('Album_ID')['Classification'].mean().reset_index()
review_scores.rename(columns={'Classification': 'avg_review_score'}, inplace=True)

# Base: medidas numéricas da fact_music
df_m = fact_music[[
    'Music_ID', 'Album_ID',
    'Duration', 'Danceability', 'Energy', 'Loudness',
    'Speechiness', 'Acousticness', 'Instrumentalness',
    'Liveness', 'Music popularity', 'Word count'
]].copy()

df_m = df_m.merge(review_scores, on='Album_ID', how='left')

def safe_float(val, decimals=4):
    """Converte para float com precisão reduzida; retorna None se inválido."""
    try:
        return round(float(val), decimals) if pd.notnull(val) else None
    except (ValueError, TypeError):
        return None

def safe_int(val):
    try:
        return int(val) if pd.notnull(val) else None
    except (ValueError, TypeError):
        return None

measure_documents = []
for _, row in df_m.iterrows():
    doc = {
        "music_id": safe_int(row['Music_ID']),
        "measures": {
            "duration_ms":       safe_int(row['Duration']),
            "danceability":      safe_float(row['Danceability']),
            "energy":            safe_float(row['Energy']),
            "loudness":          safe_float(row['Loudness']),
            "speechiness":       safe_float(row['Speechiness']),
            "acousticness":      safe_float(row['Acousticness']),
            "instrumentalness":  safe_float(row['Instrumentalness']),
            "liveness":          safe_float(row['Liveness']),
            "music_popularity":  safe_int(row['Music popularity']),
            "word_count":        safe_int(row['Word count']),
            "avg_review_score":  safe_float(row['avg_review_score'])
        }
    }
    measure_documents.append(doc)

with open('jsons/measure_documents.json', 'w', encoding='utf-8') as f:
    json.dump(measure_documents, f, indent=2, ensure_ascii=False)

print(f"Sucesso! Foram gerados {len(measure_documents)} documentos de medidas.")


Sucesso! Foram gerados 40593 documentos de medidas.


## Step 5: Text Representation and Preprocessing

In [31]:
def extract_values(obj):
    values = []
    if isinstance(obj, dict):
        for v in obj.values():
            values.extend(extract_values(v))
    elif isinstance(obj, list):
        for item in obj:
            values.extend(extract_values(item))
    else:
        values.append(str(obj))
    return values

df = pd.DataFrame({
    "doc_id": range(len(documents)),
    "text": [" ".join(extract_values(doc)) for doc in documents]
})

df.head()

,doc_id,text
0,0,I Wanna Be Yours 2013-08-01 arctic monkeys am ...
1,1,Starboy 2016-09-22 the weeknd starboy electron...
2,2,Starboy 2016-09-22 the weeknd starboy electron...
3,3,Mr. Brightside 2004-10-02 the killers hot fuss...
4,4,Mr. Brightside 2004-10-02 the killers hot fuss...


In [32]:
_CONTRACTIONS_PATH =  "jsons/contractions_en.json"
 
def _load_contractions(path: Path = _CONTRACTIONS_PATH) -> dict[str, str]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)
 

### Clean text

In [33]:
def clean_text(s: str) -> str:
    # 1. Normalizar unicode (ex: é → e, ñ → n)
    s = unicodedata.normalize("NFKD", s)

    s = s.encode("ascii", "ignore").decode("ascii")

    s = s.lower()

    s = re.sub(r"[^a-z0-9\s-]", " ", s)

    # Remover números separados
    # s = re.sub(r"\b\d+\b", " ", s)

    # Tornar múltiplos espaços em um único
    s = re.sub(r"\s+", " ", s)

    return s.strip()

 

df["text_after_processing"] = df["text"].apply(clean_text)
df.head()

,doc_id,text,text_after_processing
0,0,I Wanna Be Yours 2013-08-01 arctic monkeys am ...,i wanna be yours 2013-08-01 arctic monkeys am ...
1,1,Starboy 2016-09-22 the weeknd starboy electron...,starboy 2016-09-22 the weeknd starboy electron...
2,2,Starboy 2016-09-22 the weeknd starboy electron...,starboy 2016-09-22 the weeknd starboy electron...
3,3,Mr. Brightside 2004-10-02 the killers hot fuss...,mr brightside 2004-10-02 the killers hot fuss ...
4,4,Mr. Brightside 2004-10-02 the killers hot fuss...,mr brightside 2004-10-02 the killers hot fuss ...


### Normalize text

In [34]:

def normalize_text(s: str, contractions: dict[str, str] | None = None) -> str:
    if contractions is None:
        contractions = _load_contractions()

    for contraction, expansion in contractions.items():
        s = re.sub(r"\b" + re.escape(contraction) + r"\b", expansion, s)
 
    return s

df["text_after_processing"] = df["text_after_processing"].apply(normalize_text)
df.head()

,doc_id,text,text_after_processing
0,0,I Wanna Be Yours 2013-08-01 arctic monkeys am ...,i want to be yours 2013-08-01 arctic monkeys a...
1,1,Starboy 2016-09-22 the weeknd starboy electron...,starboy 2016-09-22 the weeknd starboy electron...
2,2,Starboy 2016-09-22 the weeknd starboy electron...,starboy 2016-09-22 the weeknd starboy electron...
3,3,Mr. Brightside 2004-10-02 the killers hot fuss...,mr brightside 2004-10-02 the killers hot fuss ...
4,4,Mr. Brightside 2004-10-02 the killers hot fuss...,mr brightside 2004-10-02 the killers hot fuss ...


### Tokenization

In [35]:
def tokenize(s: str):
    return s.split() 

df["tokens"] = df["text_after_processing"].apply(tokenize)
df.head()

,doc_id,text,text_after_processing,tokens
0,0,I Wanna Be Yours 2013-08-01 arctic monkeys am ...,i want to be yours 2013-08-01 arctic monkeys a...,"[i, want, to, be, yours, 2013-08-01, arctic, m..."
1,1,Starboy 2016-09-22 the weeknd starboy electron...,starboy 2016-09-22 the weeknd starboy electron...,"[starboy, 2016-09-22, the, weeknd, starboy, el..."
2,2,Starboy 2016-09-22 the weeknd starboy electron...,starboy 2016-09-22 the weeknd starboy electron...,"[starboy, 2016-09-22, the, weeknd, starboy, el..."
3,3,Mr. Brightside 2004-10-02 the killers hot fuss...,mr brightside 2004-10-02 the killers hot fuss ...,"[mr, brightside, 2004-10-02, the, killers, hot..."
4,4,Mr. Brightside 2004-10-02 the killers hot fuss...,mr brightside 2004-10-02 the killers hot fuss ...,"[mr, brightside, 2004-10-02, the, killers, hot..."


### Stop word removal

In [36]:
import nltk
nltk.download("stopwords")

from nltk.corpus import stopwords

stop_en = set(stopwords.words("english"))

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_en]

df["tokens_no_stop"] = df["tokens"].apply(remove_stopwords)
df[["tokens", "tokens_no_stop"]].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nunor\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


,tokens,tokens_no_stop
0,"[i, want, to, be, yours, 2013-08-01, arctic, m...","[want, 2013-08-01, arctic, monkeys, pop, want,..."
1,"[starboy, 2016-09-22, the, weeknd, starboy, el...","[starboy, 2016-09-22, weeknd, starboy, electro..."
2,"[starboy, 2016-09-22, the, weeknd, starboy, el...","[starboy, 2016-09-22, weeknd, starboy, electro..."
3,"[mr, brightside, 2004-10-02, the, killers, hot...","[mr, brightside, 2004-10-02, killers, hot, fus..."
4,"[mr, brightside, 2004-10-02, the, killers, hot...","[mr, brightside, 2004-10-02, killers, hot, fus..."


### Stemming

In [37]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer() 

def apply_stemming(tokens):
    return [stemmer.stem(t) for t in tokens]

df["stems"] = df["tokens_no_stop"].apply(apply_stemming)
df[["tokens_no_stop", "stems"]].head()


,tokens_no_stop,stems
0,"[want, 2013-08-01, arctic, monkeys, pop, want,...","[want, 2013-08-01, arctic, monkey, pop, want, ..."
1,"[starboy, 2016-09-22, weeknd, starboy, electro...","[starboy, 2016-09-22, weeknd, starboy, electro..."
2,"[starboy, 2016-09-22, weeknd, starboy, electro...","[starboy, 2016-09-22, weeknd, starboy, electro..."
3,"[mr, brightside, 2004-10-02, killers, hot, fus...","[mr, brightsid, 2004-10-02, killer, hot, fuss,..."
4,"[mr, brightside, 2004-10-02, killers, hot, fus...","[mr, brightsid, 2004-10-02, killer, hot, fuss,..."


### Join token lists back into strings so they can be vectorized.


In [38]:
def join_tokens(tokens):
    return " ".join(tokens)  

df["text_no_stop"] = df["tokens_no_stop"].apply(join_tokens)  
df["text_stemming"] = df["stems"].apply(join_tokens)  

df[["text", "text_no_stop", "text_stemming"]].head()

,text,text_no_stop,text_stemming
0,I Wanna Be Yours 2013-08-01 arctic monkeys am ...,want 2013-08-01 arctic monkeys pop want vacuum...,want 2013-08-01 arctic monkey pop want vacuum ...
1,Starboy 2016-09-22 the weeknd starboy electron...,starboy 2016-09-22 weeknd starboy electronic t...,starboy 2016-09-22 weeknd starboy electron try...
2,Starboy 2016-09-22 the weeknd starboy electron...,starboy 2016-09-22 weeknd starboy electronic t...,starboy 2016-09-22 weeknd starboy electron try...
3,Mr. Brightside 2004-10-02 the killers hot fuss...,mr brightside 2004-10-02 killers hot fuss rock...,mr brightsid 2004-10-02 killer hot fuss rock c...
4,Mr. Brightside 2004-10-02 the killers hot fuss...,mr brightside 2004-10-02 killers hot fuss rock...,mr brightsid 2004-10-02 killer hot fuss rock c...
